# CPU audit: truncated hybrid-prefill page alignment

Self-contained Python standard-library diagnostic. No model, GPU, HTTP call, package installation, game action or competition submission is performed by these cells.

This notebook executes small, unchanged method excerpts from Penny Royal 2.5.3 and 3.1.3 against controlled stub objects. It tests **the committed token-range boundary only**. It does not execute the cache consumers, tensor allocators, scheduler as a whole or a GPU kernel, and does not establish that any saved gameplay hit this case, corrupted state or lost score.

Original producer: `PrefillAdder.add_chunked_req`. Upstream correction: [Penny Royal PR 131](https://github.com/jpezzulli/sglang-rtxpro6000/pull/131), adapting SGLang [PR 45607](https://github.com/sgl-project/sglang/pull/45607). See exact revisions and hashes below.

The excerpts retain their upstream license: Apache License 2.0, copyright 2023–2024 SGLang Team. The complete license is embedded below. Modifications here are limited to extraction, dedenting and the separate diagnostic fixture; the method bodies are unchanged.

Executed outputs were captured with `compile`/`exec` in one shared Python namespace, in cell order, because nbclient/Jupyter were not installed locally. The same cells can be rerun in a normal Python notebook.

In [1]:
import hashlib, json, types
from pathlib import Path
PROVENANCE = {'pennyroyal-v2.5.3': {'source_commit': 'd00d88efc8d6281b12be4f4073126aec95038c55', 'source_url': 'https://raw.githubusercontent.com/jpezzulli/sglang-rtxpro6000/pennyroyal-v2.5.3/python/sglang/srt/managers/schedule_policy.py', 'source_file_sha256': '64f3ea4ab0ac071d56639e407abd36ea392a656c2966ed92e6614dfa85cc02e8', 'original_excerpt_sha256': {'add_chunked_req': 'd71442519fbf7dfd463251d4bdfe50fd8fbce4cc31371bdeb35b02aa1e0a413c'}, 'embedded_dedented_sha256': {'add_chunked_req': 'bd89cdcf19fcb51e3693132a8ebe907d06d0736b67d1db6b34115ae5534764e6'}}, 'pennyroyal-v3.1.3': {'source_commit': '177697b3d10fc0a4bcb17b3d84a378eecc828f68', 'source_url': 'https://raw.githubusercontent.com/jpezzulli/sglang-rtxpro6000/pennyroyal-v3.1.3/python/sglang/srt/managers/schedule_policy.py', 'source_file_sha256': '0f3dec701d3d3ecbe7199daf792020257f7cce79720db0075312d51ca99755c5', 'original_excerpt_sha256': {'_align_truncated_hybrid_chunk': '66bee0310cb05c57bd5cb7106cc23f13f3cc22c9bf5f1bbf65226470338fb78a', 'add_chunked_req': '704074e34e587f3547da7c001f662c8c6d2d4c73eef0e0e9da1530132e07d182'}, 'embedded_dedented_sha256': {'_align_truncated_hybrid_chunk': '9d95596933c4c0b79ea37ccc6d0792f86c251ae6c9948528c4c50a03f317f43f', 'add_chunked_req': 'd0ff899b17213ee4e8de5d5f136ed19884af30b075e5497ccc5ee0a5fa505d68'}}}
EXCERPTS = {'pennyroyal-v2.5.3': {'add_chunked_req': 'def add_chunked_req(self, req: Req):\n    if self.dllm_config is not None:\n        _rem_tokens = self._get_dllm_remain_tokens()\n    else:\n        _rem_tokens = min(self.rem_chunk_tokens, int(self.rem_total_tokens))\n        if self.is_hybrid_swa:\n            # alloc_extend needs extend_num_tokens + page_size per request,\n            # so reserve one page here to avoid OOM\n            _rem_tokens = min(\n                _rem_tokens, int(self.rem_swa_tokens) - self.page_size\n            )\n        # The chunked_req must be added to the list; otherwise, it will cause a memory leak.\n        # Therefore, in certain cases where _rem_tokens <= 0, it should be replaced with rem_chunk_tokens.\n        if _rem_tokens <= 0:\n            if self.is_hybrid_swa:\n                return req\n            _rem_tokens = self.rem_chunk_tokens\n\n    # A mid-chunk rank prefills this pass regardless of the delayer\n    # verdict, so report prefillable=True and ignore the result.\n    if self.prefill_delayer_single_pass is not None:\n        self.prefill_delayer_single_pass.negotiate_should_allow_prefill(\n            local_prefillable=True,\n            running_batch=self.running_batch.batch_size(),\n            max_prefill_bs=self.max_prefill_bs,\n            max_running_requests=self.max_running_requests,\n            waiting_queue_len=self.waiting_queue_len,\n        )\n\n    cand_extend_input_len = len(req.full_untruncated_fill_ids) - len(\n        req.prefix_indices\n    )\n    truncated = cand_extend_input_len > _rem_tokens\n    new_len = min(cand_extend_input_len, _rem_tokens)\n    req.set_extend_range(len(req.prefix_indices), len(req.prefix_indices) + new_len)\n    self.can_run_list.append(req)\n    self._update_prefill_budget(\n        0,\n        req.extend_range.length,\n        (\n            min(req.sampling_params.max_new_tokens, CLIP_MAX_NEW_TOKENS)\n            if not truncated\n            else 0\n        ),\n        req.retracted_stain,\n        mamba_gap_reserve=self._mamba_gap_budget_for_req(req),\n    )\n\n    # Return if chunked prefill not finished\n    return req if truncated else None\n'}, 'pennyroyal-v3.1.3': {'_align_truncated_hybrid_chunk': 'def _align_truncated_hybrid_chunk(self, prefix_len: int, chunk_len: int) -> int:\n    """Largest length that ends a *truncated* prefill chunk on a KV page\n    boundary, or 0 when not even one whole page fits.\n\n    The first chunk that ends off-page can still donate a page-aligned\n    checkpoint, because the checkpoint depth floors to the mamba grid. The\n    damage is one pass later: the tokens past that checkpoint stay private to\n    the request, so `len(req.prefix_indices)` is off-page and the NEXT chunk\'s\n    checkpoint depth is off-page too, while the key it is donated under is\n    floored to page_size. Only paged full-attention + Mamba is affected, and\n    `add_one_req` already aligns its truncated chunk.\n\n    Untruncated final chunks stay unrestricted: a request that keeps decoding\n    does not release its boundary page at commit time, so realigning the tail\n    would strand the tokens past the last checkpoint.\n    """\n    if not self.is_hybrid_ssm_cache or self.page_size == 1:\n        return chunk_len\n    end = (prefix_len + chunk_len) // self.page_size * self.page_size\n    return max(end - prefix_len, 0)\n', 'add_chunked_req': 'def add_chunked_req(self, req: Req):\n    if self.dllm_config is not None:\n        _rem_tokens = self._get_dllm_remain_tokens()\n    else:\n        _rem_tokens = min(self.rem_chunk_tokens, int(self.rem_total_tokens))\n        if self.is_hybrid_swa:\n            # alloc_extend needs extend_num_tokens + page_size per request,\n            # so reserve one page here to avoid OOM\n            _rem_tokens = min(\n                _rem_tokens, int(self.rem_swa_tokens) - self.page_size\n            )\n        # The chunked_req must be added to the list; otherwise, it will cause a memory leak.\n        # Therefore, in certain cases where _rem_tokens <= 0, it should be replaced with rem_chunk_tokens.\n        if _rem_tokens <= 0:\n            if self.is_hybrid_swa:\n                return req\n            _rem_tokens = self.rem_chunk_tokens\n\n    # A mid-chunk rank prefills this pass regardless of the delayer\n    # verdict, so report prefillable=True and ignore the result.\n    if self.prefill_delayer_single_pass is not None:\n        self.prefill_delayer_single_pass.negotiate_should_allow_prefill(\n            local_prefillable=True,\n            running_batch=self.running_batch.batch_size(),\n            max_prefill_bs=self.max_prefill_bs,\n            max_running_requests=self.max_running_requests,\n            waiting_queue_len=self.waiting_queue_len,\n        )\n\n    cand_extend_input_len = len(req.full_untruncated_fill_ids) - len(\n        req.prefix_indices\n    )\n    truncated = cand_extend_input_len > _rem_tokens\n    new_len = min(cand_extend_input_len, _rem_tokens)\n    if truncated:\n        new_len = self._align_truncated_hybrid_chunk(\n            len(req.prefix_indices), new_len\n        )\n        if new_len <= 0:\n            # No whole KV page fits: park the chunk (extend_range.end stays at\n            # len(prefix_indices), so the scheduler skips stashing it) and retry\n            # once running requests free KV. Same shape as the hybrid-SWA defer.\n            return req\n    req.set_extend_range(len(req.prefix_indices), len(req.prefix_indices) + new_len)\n    self.can_run_list.append(req)\n    self._update_prefill_budget(\n        0,\n        req.extend_range.length,\n        (\n            min(req.sampling_params.max_new_tokens, CLIP_MAX_NEW_TOKENS)\n            if not truncated\n            else 0\n        ),\n        req.retracted_stain,\n        mamba_gap_reserve=self._mamba_gap_budget_for_req(req),\n    )\n\n    # Return if chunked prefill not finished\n    return req if truncated else None\n'}}
UPSTREAM_LICENSE = '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2023-2024 SGLang Team\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n'
SCOPES = {}
for revision, methods in EXCERPTS.items():
    for name, source in methods.items():
        assert hashlib.sha256(source.encode()).hexdigest() == PROVENANCE[revision]['embedded_dedented_sha256'][name]
    scope = {'CLIP_MAX_NEW_TOKENS': 4096}
    exec('from __future__ import annotations\n' + '\n'.join(methods.values()), scope)
    SCOPES[revision] = scope
print(json.dumps(PROVENANCE, indent=2))


{
  "pennyroyal-v2.5.3": {
    "source_commit": "d00d88efc8d6281b12be4f4073126aec95038c55",
    "source_url": "https://raw.githubusercontent.com/jpezzulli/sglang-rtxpro6000/pennyroyal-v2.5.3/python/sglang/srt/managers/schedule_policy.py",
    "source_file_sha256": "64f3ea4ab0ac071d56639e407abd36ea392a656c2966ed92e6614dfa85cc02e8",
    "original_excerpt_sha256": {
      "add_chunked_req": "d71442519fbf7dfd463251d4bdfe50fd8fbce4cc31371bdeb35b02aa1e0a413c"
    },
    "embedded_dedented_sha256": {
      "add_chunked_req": "bd89cdcf19fcb51e3693132a8ebe907d06d0736b67d1db6b34115ae5534764e6"
    }
  },
  "pennyroyal-v3.1.3": {
    "source_commit": "177697b3d10fc0a4bcb17b3d84a378eecc828f68",
    "source_url": "https://raw.githubusercontent.com/jpezzulli/sglang-rtxpro6000/pennyroyal-v3.1.3/python/sglang/srt/managers/schedule_policy.py",
    "source_file_sha256": "0f3dec701d3d3ecbe7199daf792020257f7cce79720db0075312d51ca99755c5",
    "original_excerpt_sha256": {
      "_align_truncated_hybrid_chu

## Controlled method fixture

The fixture supplies only fields exercised by these methods. Budget accounting and Mamba-gap callbacks are no-ops; the KV budget is an explicit input. No cache implementation is mocked as “correct” or tested indirectly. `is_hybrid_ssm_cache=True` and page size 64 are test conditions, not a statement about every runtime launch.

A parked continuation retains its previous `extend_range` and is not appended to the runnable list. A final untruncated tail is permitted to end off-page. Non-hybrid requests and page size 1 should preserve their original boundaries.

In [2]:
def run_case(revision, *, budget=123, page_size=64, hybrid=True, prefix=4096, fill=12288):
    scope = SCOPES[revision]
    obj = types.SimpleNamespace(
        dllm_config=None, rem_chunk_tokens=4096, rem_total_tokens=budget,
        is_hybrid_swa=False, is_hybrid_ssm_cache=hybrid, page_size=page_size,
        prefill_delayer_single_pass=None, can_run_list=[],
        _update_prefill_budget=lambda *args, **kwargs: None,
        _mamba_gap_budget_for_req=lambda request: 0,
    )
    if '_align_truncated_hybrid_chunk' in scope:
        obj._align_truncated_hybrid_chunk = types.MethodType(scope['_align_truncated_hybrid_chunk'], obj)
    req = types.SimpleNamespace(
        full_untruncated_fill_ids=list(range(fill)), prefix_indices=list(range(prefix)),
        sampling_params=types.SimpleNamespace(max_new_tokens=128), retracted_stain=None,
        extend_range=types.SimpleNamespace(start=0, end=prefix, length=prefix),
    )
    writes = []
    def set_range(start, end):
        writes.append([start, end])
        req.extend_range = types.SimpleNamespace(start=start, end=end, length=end-start)
    req.set_extend_range = set_range
    result = scope['add_chunked_req'](obj, req)
    return {
        'revision': revision, 'prefix': prefix, 'remaining_kv_budget': budget,
        'page_size': page_size, 'hybrid': hybrid,
        'range_writes': writes, 'final_range_end': req.extend_range.end,
        'end_mod_page': req.extend_range.end % page_size,
        'runnable_count': len(obj.can_run_list), 'continuation_returned': result is req,
    }


In [3]:
OLD, NEW = 'pennyroyal-v2.5.3', 'pennyroyal-v3.1.3'
RESULTS = {}
for name, options in [
    ('kv_budget_123_hybrid_page64', {}),
    ('kv_budget_63_below_one_page', {'budget': 63}),
    ('non_hybrid_123', {'hybrid': False}),
    ('page_size_one', {'page_size': 1}),
    ('untruncated_final_tail', {'fill': 4118}),
]:
    RESULTS[name] = {'old': run_case(OLD, **options), 'new': run_case(NEW, **options)}
main = RESULTS['kv_budget_123_hybrid_page64']
assert main['old']['final_range_end'] == 4219 and main['old']['end_mod_page'] == 59
assert main['new']['final_range_end'] == 4160 and main['new']['end_mod_page'] == 0
small = RESULTS['kv_budget_63_below_one_page']
assert small['old']['final_range_end'] == 4159 and small['old']['runnable_count'] == 1
assert small['new']['final_range_end'] == 4096 and small['new']['runnable_count'] == 0
assert small['new']['range_writes'] == [] and small['new']['continuation_returned']
for name in ['non_hybrid_123', 'page_size_one']:
    assert RESULTS[name]['old']['final_range_end'] == RESULTS[name]['new']['final_range_end'] == 4219
for side in ['old', 'new']:
    assert RESULTS['untruncated_final_tail'][side]['final_range_end'] == 4118
    assert not RESULTS['untruncated_final_tail'][side]['continuation_returned']
print(json.dumps(RESULTS, indent=2))


{
  "kv_budget_123_hybrid_page64": {
    "old": {
      "revision": "pennyroyal-v2.5.3",
      "prefix": 4096,
      "remaining_kv_budget": 123,
      "page_size": 64,
      "hybrid": true,
      "range_writes": [
        [
          4096,
          4219
        ]
      ],
      "final_range_end": 4219,
      "end_mod_page": 59,
      "runnable_count": 1,
      "continuation_returned": true
    },
    "new": {
      "revision": "pennyroyal-v3.1.3",
      "prefix": 4096,
      "remaining_kv_budget": 123,
      "page_size": 64,
      "hybrid": true,
      "range_writes": [
        [
          4096,
          4160
        ]
      ],
      "final_range_end": 4160,
      "end_mod_page": 0,
      "runnable_count": 1,
      "continuation_returned": true
    }
  },
  "kv_budget_63_below_one_page": {
    "old": {
      "revision": "pennyroyal-v2.5.3",
      "prefix": 4096,
      "remaining_kv_budget": 63,
      "page_size": 64,
      "hybrid": true,
      "range_writes": [
        [
          4

In [4]:
SUMMARY = {
    'scope': 'CPU method-boundary fixture; not full cache-state or GPU validation',
    'cases_passed': len(RESULTS),
    'old_end_at_budget_123': main['old']['final_range_end'],
    'new_end_at_budget_123': main['new']['final_range_end'],
    'new_subpage_budget_parked': small['new']['runnable_count'] == 0,
    'gpu_operations': 0, 'model_loads': 0, 'network_requests': 0,
    'score_evidence': False,
    'conclusion': 'The unaligned producer path exists in the pinned 2.5.3 source; 3.1.3 changes its boundary under the tested hybrid/page conditions. Actual gameplay impact remains unmeasured.',
    'provenance': PROVENANCE,
}
Path('arc3_cpu_prefill_alignment_audit.json').write_text(json.dumps(SUMMARY, indent=2) + '\n')
print(json.dumps(SUMMARY, indent=2))


{
  "scope": "CPU method-boundary fixture; not full cache-state or GPU validation",
  "cases_passed": 5,
  "old_end_at_budget_123": 4219,
  "new_end_at_budget_123": 4160,
  "new_subpage_budget_parked": true,
  "gpu_operations": 0,
  "model_loads": 0,
  "network_requests": 0,
  "score_evidence": false,
  "conclusion": "The unaligned producer path exists in the pinned 2.5.3 source; 3.1.3 changes its boundary under the tested hybrid/page conditions. Actual gameplay impact remains unmeasured.",
  "provenance": {
    "pennyroyal-v2.5.3": {
      "source_commit": "d00d88efc8d6281b12be4f4073126aec95038c55",
      "source_url": "https://raw.githubusercontent.com/jpezzulli/sglang-rtxpro6000/pennyroyal-v2.5.3/python/sglang/srt/managers/schedule_policy.py",
      "source_file_sha256": "64f3ea4ab0ac071d56639e407abd36ea392a656c2966ed92e6614dfa85cc02e8",
      "original_excerpt_sha256": {
        "add_chunked_req": "d71442519fbf7dfd463251d4bdfe50fd8fbce4cc31371bdeb35b02aa1e0a413c"
      },
      "em

## Interpretation and next gate

The old method can commit an off-page continuing chunk under a non-page-sized remaining KV budget. The corrected method ends that truncated hybrid chunk on a complete page and parks it when no complete page fits. Both preserve the tested untruncated tail and non-hybrid/page-one cases.

This makes the old source path relevant to an applicability audit. It is **not** an end-to-end reproduction of cache-state corruption, proof that a specific game encountered the condition, an engine backport qualification, a speed measurement or a score improvement. Check the actual runtime cache class, exercised budget traces and downstream state/key invariants before applying or comparing a serving change.